<a href="https://colab.research.google.com/github/parthibray2002/Optimizing-IPL-Auctions-using-Gen-AI-ML-/blob/main/MS_Implementation_Diffusion_Model_Data_Generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from google.colab import drive, files

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount = True)

Mounted at /content/drive


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
file_path = '/content/drive/My Drive/Upgrad MS Dataset/IPL Auction Dataset.xlsx'

# 3. Read the 4th sheet (index 3)
df = pd.read_excel(file_path, sheet_name = 'All Rounders' )

print(df)

                Player       Category      Country              Role  \
0        Hardik Pandya  International        India  Fast-All Rounder   
1      Ravindra Jadeja  International        India  Spin-All Rounder   
2          Shivam Dube  International        India  Fast-All Rounder   
3   Nitish Kumar Reddy  International        India  Fast-All Rounder   
4           Axar Patel  International        India  Spin-All Rounder   
..                 ...            ...          ...               ...   
69       Gareth Delany  International      Ireland  Fast-All Rounder   
70        Bas de Leede  International  Netherlands  Fast-All Rounder   
71     Colin Ackermann  International  Netherlands  Spin All Rounder   
72      Logan van Beek  International  Netherlands  Fast-All Rounder   
73      Saqib Zulfiqar  International  Netherlands  Spin All Rounder   

    Runs Scored  Batting Strike Rate  Batting Average  No of Sixes  \
0          2757                147.0        28.340000          14

In [ ]:
num_cols = ['Runs Scored', 'Batting Strike Rate', 'Batting Average', 'No of Sixes',
            'No of Fours', 'Balls Faced', 'Total career dot balls played',
            'No of Centuries Scored', 'No of Half Centuries Scored', 'Wickets Taken',
            '3 Wicket Hauls', '5 Wicket Hauls', 'Bowling Strike Rate', 'Bowling Economy',
            'Bowling Average', 'Runs Conceded', 'Sixes Conceded', 'Fours Conceded',
            'Dot Balls Bowled', 'Maidens Bowled', 'Age of the Player', 'Last Auction Price Sold (in Crs)']

cat_cols = ['Category', 'Country', 'Role', 'Capped/Uncapped']

In [ ]:
df_log = df.copy()
for col in num_cols:
    df_log[col] = np.log1p(df_log[col]) # log(1 + x)

cat_cols = ['Category', 'Country', 'Role', 'Capped/Uncapped']

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), cat_cols)
])

x_real = torch.FloatTensor(preprocessor.fit_transform(df_log))
input_dim = x_real.shape[1]

In [ ]:
class DenoiseNet(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim + 1, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, dim)
        )
    def forward(self, x, t):
        return self.net(torch.cat([x, t.view(-1, 1)], dim=1))

In [ ]:
T = 1000
alphas = torch.linspace(0.9999, 0.98, T)
alphas_cumprod = torch.cumprod(alphas, dim=0)

model = DenoiseNet(input_dim)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print("Training Diffusion Model...")
model.train()
for epoch in range(1200):
    t = torch.randint(0, T, (x_real.shape[0],))
    noise = torch.randn_like(x_real)
    a_cp = alphas_cumprod[t].view(-1, 1)
    x_noisy = torch.sqrt(a_cp) * x_real + torch.sqrt(1 - a_cp) * noise

    predicted_noise = model(x_noisy, t.float() / T)
    loss = nn.functional.mse_loss(predicted_noise, noise)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

Training Diffusion Model...


In [ ]:
model.eval()
with torch.no_grad():
    x_gen = torch.randn(200, input_dim)
    for t_idx in reversed(range(T)):
        t_tensor = torch.full((200,), t_idx / T)
        noise_pred = model(x_gen, t_tensor)
        a = alphas[t_idx]
        a_cp = alphas_cumprod[t_idx]
        z = torch.randn_like(x_gen) if t_idx > 0 else 0
        x_gen = (1 / torch.sqrt(a)) * (x_gen - ((1 - a) / torch.sqrt(1 - a_cp)) * noise_pred) + torch.sqrt(1 - a) * z

In [ ]:
x_gen_np = x_gen.numpy()

# Get the individual transformers from the preprocessor
num_transformer = preprocessor.named_transformers_['num']
cat_transformer = preprocessor.named_transformers_['cat']

# Determine the number of numerical features
num_features_count = len(num_cols)

# Split the generated data back into numerical and one-hot encoded categorical parts
gen_num_scaled = x_gen_np[:, :num_features_count]
gen_cat_onehot = x_gen_np[:, num_features_count:]

# Inverse transform numerical features
gen_num_original = num_transformer.inverse_transform(gen_num_scaled)

# Inverse transform categorical features
gen_cat_original = cat_transformer.inverse_transform(gen_cat_onehot)

# Combine the inverse transformed data. Ensure the order matches the original columns.
combined_original_data = np.hstack((gen_num_original, gen_cat_original))

gen_df = pd.DataFrame(combined_original_data, columns=num_cols + cat_cols)

# Critical Step: Reverse the Log and Clip to zero
for col in num_cols:
    gen_df[col] = np.expm1(gen_df[col].astype(float)) # exp(x) - 1
    gen_df[col] = gen_df[col].clip(lower=0) # Final safety check

# Rounding for realism (you can't have 0.4 centuries)
int_cols = ['No of Centuries Scored', 'No of Half Centuries Scored', 'Wickets Taken', '3 Wicket Hauls', '5 Wicket Hauls']
for col in int_cols:
    gen_df[col] = gen_df[col].round().astype(int)

gen_df.insert(0, 'Player', [f"Diff_Player_{i+1}" for i in range(200)])
gen_df.to_csv('allrounders_diffusion_players_200.csv', index=False)
files.download('allrounders_diffusion_players_200.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
file_path = '/content/drive/My Drive/Upgrad MS Dataset/IPL Auction Dataset.xlsx'

# 3. Read the 4th sheet (index 3)
df = pd.read_excel(file_path, sheet_name = 'Batsmen' )

In [ ]:
num_cols = ['Batting Strike Rate', 'Batsmen Total Runs','Batting Average', 'No of Sixes', 'No of Fours', 'Balls Faced', 'Total career dot balls played','No of Centuries Scored', 'No of Half Centuries Scored','Age of the Player','Last Auction Price Sold (in Crs)']
cat_cols = ['Category', 'Country', 'Role', 'Capped or Uncapped']

In [ ]:
df_log = df.copy()
for col in num_cols:
    df_log[col] = np.log1p(df_log[col]) # log(1 + x)

cat_cols = ['Category', 'Country', 'Role', 'Capped or Uncapped']

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), cat_cols)
])

x_real = torch.FloatTensor(preprocessor.fit_transform(df_log))
input_dim = x_real.shape[1]

In [ ]:
class DenoiseNet(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim + 1, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, dim)
        )
    def forward(self, x, t):
        return self.net(torch.cat([x, t.view(-1, 1)], dim=1))

In [ ]:
T = 1000
alphas = torch.linspace(0.9999, 0.98, T)
alphas_cumprod = torch.cumprod(alphas, dim=0)

model = DenoiseNet(input_dim)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print("Training Diffusion Model...")
model.train()
for epoch in range(1200):
    t = torch.randint(0, T, (x_real.shape[0],))
    noise = torch.randn_like(x_real)
    a_cp = alphas_cumprod[t].view(-1, 1)
    x_noisy = torch.sqrt(a_cp) * x_real + torch.sqrt(1 - a_cp) * noise

    predicted_noise = model(x_noisy, t.float() / T)
    loss = nn.functional.mse_loss(predicted_noise, noise)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

Training Diffusion Model...


In [ ]:
model.eval()
with torch.no_grad():
    x_gen = torch.randn(200, input_dim)
    for t_idx in reversed(range(T)):
        t_tensor = torch.full((200,), t_idx / T)
        noise_pred = model(x_gen, t_tensor)
        a = alphas[t_idx]
        a_cp = alphas_cumprod[t_idx]
        z = torch.randn_like(x_gen) if t_idx > 0 else 0
        x_gen = (1 / torch.sqrt(a)) * (x_gen - ((1 - a) / torch.sqrt(1 - a_cp)) * noise_pred) + torch.sqrt(1 - a) * z

In [ ]:
x_gen_np = x_gen.numpy()

# Get the individual transformers from the preprocessor
num_transformer = preprocessor.named_transformers_['num']
cat_transformer = preprocessor.named_transformers_['cat']

# Determine the number of numerical features
num_features_count = len(num_cols)

# Split the generated data back into numerical and one-hot encoded categorical parts
gen_num_scaled = x_gen_np[:, :num_features_count]
gen_cat_onehot = x_gen_np[:, num_features_count:]

# Inverse transform numerical features
gen_num_original = num_transformer.inverse_transform(gen_num_scaled)

# Inverse transform categorical features
gen_cat_original = cat_transformer.inverse_transform(gen_cat_onehot)

# Combine the inverse transformed data. Ensure the order matches the original columns.
combined_original_data = np.hstack((gen_num_original, gen_cat_original))

gen_df = pd.DataFrame(combined_original_data, columns=num_cols + cat_cols)

# Critical Step: Reverse the Log and Clip to zero
for col in num_cols:
    gen_df[col] = np.expm1(gen_df[col].astype(float)) # exp(x) - 1
    gen_df[col] = gen_df[col].clip(lower=0) # Final safety check

# Rounding for realism (you can't have 0.4 centuries)
int_cols = ['Batsmen Total Runs', 'No of Sixes', 'No of Fours', 'Balls Faced', 'Total career dot balls played','No of Centuries Scored', 'No of Half Centuries Scored','Age of the Player']
for col in int_cols:
    gen_df[col] = gen_df[col].round().astype(int)

gen_df.insert(0, 'Player', [f"Diff_Player_{i+1}" for i in range(200)])
gen_df.to_csv('Batsmen_diffusion_players_200.csv', index=False)
files.download('Batsmen_diffusion_players_200.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
file_path = '/content/drive/My Drive/Upgrad MS Dataset/IPL Auction Dataset.xlsx'

# 3. Read the 4th sheet (index 3)
df_1 = pd.read_excel(file_path, sheet_name = 'Bowlers' )

In [ ]:
print(df_1)

                 Player       Category     Country                Role  \
0        Jasprit Bumrah  International       India         Pace Bowler   
1        Mohammad Siraj  International       India         Pace Bowler   
2        Mohammad Shami  International       India         Pace Bowler   
3        Arshdeep Singh  International       India         Pace Bowler   
4         Kuldeep Yadav  International       India         Spin Bowler   
..                  ...            ...         ...                 ...   
102      Rishad Hossain  International  Bangladesh         Spin Bowler   
103  Sandeep Lamichhane  International       Nepal         Spin Bowler   
104          Sher Malla  International       Nepal         Spin Bowler   
105         Sompal Kami  International       Nepal  Medium Pace Bowler   
106            Karan KC  International       Nepal         Pace Bowler   

     Bowling Strike Rate  Bowling Economy  Bowling Average  Wickets Taken  \
0                   5.50          

In [ ]:
num_cols = ['Bowling Strike Rate', 'Bowling Economy','Bowling Average', 'Wickets Taken', 'Runs Conceded', 'Sixes Conceded', 'Fours Conceded','Dot Balls Bowled', '3 Wicket Hauls','5 Wicket hauls','Maidens Bowled','Age of the Player','Last Auction Price Sold (in Crs)']
cat_cols = ['Category', 'Country', 'Role', 'Capped/Uncapped']

In [ ]:
df_log_1 = df_1.copy()
for col in num_cols:
    df_log_1[col] = np.log1p(df_log_1[col]) # log(1 + x)

cat_cols = ['Category', 'Country', 'Role', 'Capped/Uncapped']

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), cat_cols)
])

x_real = torch.FloatTensor(preprocessor.fit_transform(df_log_1))
input_dim = x_real.shape[1]

In [ ]:
class DenoiseNet(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim + 1, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, dim)
        )
    def forward(self, x, t):
        return self.net(torch.cat([x, t.view(-1, 1)], dim=1))

In [ ]:
T = 1000
alphas = torch.linspace(0.9999, 0.98, T)
alphas_cumprod = torch.cumprod(alphas, dim=0)

model = DenoiseNet(input_dim)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print("Training Diffusion Model...")
model.train()
for epoch in range(1200):
    t = torch.randint(0, T, (x_real.shape[0],))
    noise = torch.randn_like(x_real)
    a_cp = alphas_cumprod[t].view(-1, 1)
    x_noisy = torch.sqrt(a_cp) * x_real + torch.sqrt(1 - a_cp) * noise

    predicted_noise = model(x_noisy, t.float() / T)
    loss = nn.functional.mse_loss(predicted_noise, noise)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

Training Diffusion Model...


In [ ]:
model.eval()
with torch.no_grad():
    x_gen = torch.randn(200, input_dim)
    for t_idx in reversed(range(T)):
        t_tensor = torch.full((200,), t_idx / T)
        noise_pred = model(x_gen, t_tensor)
        a = alphas[t_idx]
        a_cp = alphas_cumprod[t_idx]
        z = torch.randn_like(x_gen) if t_idx > 0 else 0
        x_gen = (1 / torch.sqrt(a)) * (x_gen - ((1 - a) / torch.sqrt(1 - a_cp)) * noise_pred) + torch.sqrt(1 - a) * z

In [ ]:
x_gen_np = x_gen.numpy()

# Get the individual transformers from the preprocessor
num_transformer = preprocessor.named_transformers_['num']
cat_transformer = preprocessor.named_transformers_['cat']

# Determine the number of numerical features
num_features_count = len(num_cols)

# Split the generated data back into numerical and one-hot encoded categorical parts
gen_num_scaled = x_gen_np[:, :num_features_count]
gen_cat_onehot = x_gen_np[:, num_features_count:]

# Inverse transform numerical features
gen_num_original = num_transformer.inverse_transform(gen_num_scaled)

# Inverse transform categorical features
gen_cat_original = cat_transformer.inverse_transform(gen_cat_onehot)

# Combine the inverse transformed data. Ensure the order matches the original columns.
combined_original_data = np.hstack((gen_num_original, gen_cat_original))

gen_df = pd.DataFrame(combined_original_data, columns=num_cols + cat_cols)

# Critical Step: Reverse the Log and Clip to zero
for col in num_cols:
    gen_df[col] = np.expm1(gen_df[col].astype(float)) # exp(x) - 1
    gen_df[col] = gen_df[col].clip(lower=0) # Final safety check

# Rounding for realism (you can't have 0.4 centuries)
int_cols = ['Wickets Taken', 'Runs Conceded', 'Sixes Conceded', 'Fours Conceded','Dot Balls Bowled', '3 Wicket Hauls','5 Wicket hauls','Maidens Bowled','Age of the Player']
for col in int_cols:
    gen_df[col] = gen_df[col].round().astype(int)

gen_df.insert(0, 'Player', [f"Diff_Player_{i+1}" for i in range(200)])
gen_df.to_csv('Bowlers_diffusion_players_200.csv', index=False)
files.download('Bowlers_diffusion_players_200.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
file_path = '/content/drive/My Drive/Upgrad MS Dataset/IPL Auction Dataset.xlsx'

# 3. Read the 4th sheet (index 3)
df_2 = pd.read_excel(file_path, sheet_name = 'Wicketkeepers' )

In [ ]:
print(df_2)

                    Player       Category       Country        Role  \
0             Ishan Kishan  International         India  Wk-Batsman   
1             Sanju Samson  International         India  Wk-Batsman   
2           Abhishek Porel       Domestic         India  Wk-Batsman   
3              Dhruv Jurel  International         India  Wk-Batsman   
4               Robin Minz       Domestic         India  Wk-Batsman   
5             Rishabh Pant  International         India  Wk-Batsman   
6                 KL Rahul  International         India  Wk-Batsman   
7            Jitesh Sharma  International         India  Wk-Batsman   
8         Prabhsiman Singh       Domestic         India  Wk-Batsman   
9               Anuj Rawat       Domestic         India  Wk-Batsman   
10               KS Bharat       Domestic         India  Wk-Batsman   
11          Kumar Kushagra       Domestic         India  Wk-Batsman   
12             Aryan Juyal       Domestic         India  Wk-Batsman   
13    

In [ ]:
num_cols = ['Batting Strike Rate', 'Batsmen Total Runs','Batting Average', 'No of Sixes', 'No of Fours', 'Balls Faced', 'Total career dot balls played','No of Centuries Scored', 'No of Half Centuries Scored','No of Catches','No of Stumpings','No of Run Outs','Age of the Player','Last Auction Price Sold (in Crs)']
cat_cols = ['Category', 'Country', 'Capped or Uncapped']

In [ ]:
df_log_2 = df_2.copy()
for col in num_cols:
    df_log_2[col] = np.log1p(df_log_2[col]) # log(1 + x)

cat_cols = ['Category', 'Country', 'Capped or Uncapped']

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), cat_cols)
])

x_real = torch.FloatTensor(preprocessor.fit_transform(df_log_2))
input_dim = x_real.shape[1]

In [ ]:
class DenoiseNet(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim + 1, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, dim)
        )
    def forward(self, x, t):
        return self.net(torch.cat([x, t.view(-1, 1)], dim=1))

In [ ]:
T = 1000
alphas = torch.linspace(0.9999, 0.98, T)
alphas_cumprod = torch.cumprod(alphas, dim=0)

model = DenoiseNet(input_dim)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print("Training Diffusion Model...")
model.train()
for epoch in range(1200):
    t = torch.randint(0, T, (x_real.shape[0],))
    noise = torch.randn_like(x_real)
    a_cp = alphas_cumprod[t].view(-1, 1)
    x_noisy = torch.sqrt(a_cp) * x_real + torch.sqrt(1 - a_cp) * noise

    predicted_noise = model(x_noisy, t.float() / T)
    loss = nn.functional.mse_loss(predicted_noise, noise)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

Training Diffusion Model...


In [ ]:
x_gen_np = x_gen.numpy()

# Get the individual transformers from the preprocessor
num_transformer = preprocessor.named_transformers_['num']
cat_transformer = preprocessor.named_transformers_['cat']

# Determine the number of numerical features
num_features_count = len(num_cols)

# Split the generated data back into numerical and one-hot encoded categorical parts
gen_num_scaled = x_gen_np[:, :num_features_count]
gen_cat_onehot = x_gen_np[:, num_features_count:]

# Inverse transform numerical features
gen_num_original = num_transformer.inverse_transform(gen_num_scaled)

# Inverse transform categorical features
gen_cat_original = cat_transformer.inverse_transform(gen_cat_onehot)

# Combine the inverse transformed data. Ensure the order matches the original columns.
combined_original_data = np.hstack((gen_num_original, gen_cat_original))

gen_df = pd.DataFrame(combined_original_data, columns=num_cols + cat_cols)

# Critical Step: Reverse the Log and Clip to zero
for col in num_cols:
    gen_df[col] = np.expm1(gen_df[col].astype(float)) # exp(x) - 1
    gen_df[col] = gen_df[col].clip(lower=0) # Final safety check

# Rounding for realism (you can't have 0.4 centuries)
int_cols = ['Batsmen Total Runs','No of Sixes', 'No of Fours', 'Balls Faced', 'Total career dot balls played','No of Centuries Scored', 'No of Half Centuries Scored','No of Catches','No of Stumpings','No of Run Outs','Age of the Player']
for col in int_cols:
    gen_df[col] = gen_df[col].round().astype(int)

gen_df.insert(0, 'Player', [f"Diff_Player_{i+1}" for i in range(200)])
gen_df.to_csv('Wicketkeepers_diffusion_players_200.csv', index=False)
files.download('Wicketkeepers_diffusion_players_200.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>